<a href="https://colab.research.google.com/github/ahmadhrba/chess-transcript-validation/blob/main/Chess_Game_Reconstruction_Validator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Workflow

1. Reconstruct a sequence of chess moves from a game record.
2. Validate the sequence programmatically using `python-chess`.
3. Detect illegal or inconsistent moves in a corrupted transcript.
4. Flag the detected error for expert review rather than automatically assuming a correction.
5. Apply a verified correction and revalidate the reconstructed game.


In [12]:
!pip install python-chess

In [13]:
import chess
import chess.pgn

print("python-chess installed successfully!")
print("Version:", chess.__version__)

python-chess installed successfully!
Version: 1.11.2


## Test 1 — Valid Move Sequence
**Objective:** Validate the first 10 moves of the Leko–Erigaisi game and confirm that the complete sequence is legal.

**Expected result:** All 20 half-moves are accepted and the game is successfully validated.

In [14]:
game_1_moves = [
    "e4", "e5",
    "Nf3", "Nc6",
    "Bb5", "a6",
    "Ba4", "Nf6",
    "O-O", "Be7",
    "Re1", "b5",
    "Bb3", "d6",
    "c3", "O-O",
    "h3"
]

print("Number of half-moves:", len(game_1_moves))
print("Number of completed moves:", len(game_1_moves) // 2)

Number of half-moves: 17
Number of completed moves: 8


In [15]:
def validate_game(moves):
    board = chess.Board()

    for index, move_text in enumerate(moves):
        try:
            move = board.parse_san(move_text)
            board.push(move)
            print(f"✓ {index + 1}: {move_text}")

        except ValueError:
            print(f"✗ ERROR at move {index + 1}: {move_text}")
            return False

    print("\n✓ Game successfully validated")
    return True

In [16]:
validate_game(game_1_moves)

✓ 1: e4
✓ 2: e5
✓ 3: Nf3
✓ 4: Nc6
✓ 5: Bb5
✓ 6: a6
✓ 7: Ba4
✓ 8: Nf6
✓ 9: O-O
✓ 10: Be7
✓ 11: Re1
✓ 12: b5
✓ 13: Bb3
✓ 14: d6
✓ 15: c3
✓ 16: O-O
✓ 17: h3

✓ Game successfully validated


True

In [17]:
### Leko, Peter vs. Erigaisi, Arjun
##46th Chess Olympiad 2026 — Ruy Lopez, Classical (Cordel) Defence##

leko_erigaisi_moves = [
    "e4", "e5",
    "Nf3", "Nc6",
    "Bb5", "Bc5",
    "O-O", "d6",
    "c3", "a6",
    "Ba4", "Bg4",
    "h3", "Bh5",
    "d3", "Ne7",
    "Be3", "O-O",
    "Nbd2", "Kh8"
]

validate_game(leko_erigaisi_moves)

✓ 1: e4
✓ 2: e5
✓ 3: Nf3
✓ 4: Nc6
✓ 5: Bb5
✓ 6: Bc5
✓ 7: O-O
✓ 8: d6
✓ 9: c3
✓ 10: a6
✓ 11: Ba4
✓ 12: Bg4
✓ 13: h3
✓ 14: Bh5
✓ 15: d3
✓ 16: Ne7
✓ 17: Be3
✓ 18: O-O
✓ 19: Nbd2
✓ 20: Kh8

✓ Game successfully validated


True

## Test 2 — Detecting a Transcript Error

To simulate a transcription mistake, I will change a legal move from the game and make it illegal.
The validator should identify the first move that is inconsistent with the legal board position.

In [18]:
leko_erigaisi_error = [
    "e4", "e5",
    "Nf3", "Nc6",
    "Bb5", "Bc5",
    "O-O", "d6",
    "c3", "a6",
    "Ba4", "Bg5",   # transcription error
    "h3", "Bh5",
    "d3", "Ne7",
    "Be3", "O-O",
    "Nbd2", "Kh8"
]

validate_game(leko_erigaisi_error)

✓ 1: e4
✓ 2: e5
✓ 3: Nf3
✓ 4: Nc6
✓ 5: Bb5
✓ 6: Bc5
✓ 7: O-O
✓ 8: d6
✓ 9: c3
✓ 10: a6
✓ 11: Ba4
✗ ERROR at move 12: Bg5


False

##TRANSCRIPT VALIDATION REPORT

Status: ERROR DETECTED

Position: black move N6...Bg5

Problem: illegal move

Last valid move: white move N6.Ba4

Moves successfully reconstructed: 11

## Test 3 — Transcript Correction By Expert Chess Player

**Question:** Can we determine what the transcript probably meant?

**Objective:** Review the illegal move detected in Test 2, identify the likely intended move using chess expertise and the verified game record, apply the correction, and validate the reconstructed sequence.

**Detected error:** `6...Bg5`

**Verified correction:** `6...Bg4`

**Expected result:** After replacing the incorrect move, the reconstructed sequence should pass validation.

In [19]:
# Copy the erroneous transcript
corrected_moves = leko_erigaisi_error.copy()

# Correct the detected error:
# Index 11 = Black's 6th move
corrected_moves[11] = "Bg4"

print("Correction applied:")
print("6...Bg5  →  6...Bg4")

print("\nValidating corrected game:")
validate_game(corrected_moves)

Correction applied:
6...Bg5  →  6...Bg4

Validating corrected game:
✓ 1: e4
✓ 2: e5
✓ 3: Nf3
✓ 4: Nc6
✓ 5: Bb5
✓ 6: Bc5
✓ 7: O-O
✓ 8: d6
✓ 9: c3
✓ 10: a6
✓ 11: Ba4
✓ 12: Bg4
✓ 13: h3
✓ 14: Bh5
✓ 15: d3
✓ 16: Ne7
✓ 17: Be3
✓ 18: O-O
✓ 19: Nbd2
✓ 20: Kh8

✓ Game successfully validated


True

**Note to mention:** The validator detects errors but does not silently invent corrections. Invalid transcript entries are flagged for expert review before the corrected sequence is revalidated.

## Project Summary

This project demonstrates a simple workflow for chess game reconstruction and transcript quality assurance using Python and chess expertise.
## Test Game

**Peter Leko vs. Arjun Erigaisi**  
46th Chess Olympiad 2026, Samarkand  
Ruy Lopez, Classical (Cordel) Defence

## Reviewer

**Ahmad Harba**  
FIDE Classical: 1903  
FIDE Rapid: 1903  
FIDE Blitz: 1953